# Costs, calibration, and held-out routing

Demonstration documents only. These notebooks inspect the committed evidence; they do not claim a separate full-dataset training run.

In [1]:
from pathlib import Path
import json
import pandas as pd
root = Path.cwd()
e = json.loads((root / 'results/manifest.json').read_text(encoding='utf-8'))
print('Population:', e['scope'])


Population: {'generated': {'train': 48, 'validation': 24, 'test': 24}, 'cord': {'train': 48, 'validation': 20, 'test': 20}, 'limitations': ['Bounded CORD subset, not the full benchmark: first-shard seeded training sample and first validation/test rows.', 'Generated invoices share four templates; performance does not establish generalization to unseen vendors or layouts.', 'Small test populations give wide intervals; the chosen costs are illustrative (review 1, wrong approval 20).', 'CORD gold words cover annotated semantic lines; full-page OCR CER/WER includes unannotated text.', 'Gold/real OCR input gaps include token segmentation and layout conventions, so they are not pure additive character-recognition loss.', 'Rules are conservative on receipts without invoice date, vendor and PO evidence, so those receipts go to review.', 'Other-class classification uses generated text tokens; no real miscellaneous mailroom documents were evaluated.', 'Extraction measurements condition on the ref

In [2]:
op = e['operating_point']
print({key:value for key,value in op.items() if key != 'curve'})
print(pd.DataFrame(op['curve']).to_string(index=False))
calibration = [{key:value for key,value in row.items() if key != 'bins'} for row in e['calibration']]
print(pd.DataFrame(calibration).to_string(index=False))
assert op['chosen_on'] == 'validation'
assert op['reported_on'] == 'test'

{'threshold': 1.0, 'chosen_on': 'validation', 'review_cost': 1, 'error_cost': 20, 'validation_n': 44, 'reported_on': 'test', 'backend': 'rules', 'ocr_engine': 'tesseract', 'n': 44, 'auto_approved': 14, 'review_rate': {'value': 0.6818181818181818, 'ci': [0.5344310393781817, 0.800006922891463], 'n': 44, 'method': '95% Wilson score interval'}, 'auto_approval_error': {'value': 0.0, 'ci': [0, 0.2153108027376265], 'n': 14, 'method': '95% Wilson score interval'}, 'correctness': 'All required normalized fields correct and no planted business defect'}
 threshold  review_rate  error_rate  auto_approved  wrong_auto_approvals     cost
  0.000000     0.681818         0.0             14                     0 0.681818
  0.050000     0.681818         0.0             14                     0 0.681818
  0.100000     0.681818         0.0             14                     0 0.681818
  0.150000     0.681818         0.0             14                     0 0.681818
  0.200000     0.681818         0.0      

## Rejected shortcut / negative control

In [3]:
# Negative control: test-selected routing must be rejected by publication.
from copy import deepcopy
from scripts.render_reports import validate_manifest
invalid = deepcopy(e)
invalid['operating_point']['chosen_on'] = 'test'
try:
    validate_manifest(invalid)
except ValueError as error:
    print('Rejected:', error)
else:
    raise AssertionError('Test leakage was not rejected')

Rejected: Routing must be selected on validation and reported on test
